In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             477570
   Errors:                    521
   Search Artists:            1803604
   Known Summary IDs:         718650


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [45]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  1162484
#   Artist Names To Get:  1153134
#   Artist Names To Get:  1138934
#   Artist Names To Get:  1120584

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   694220
#   Artist Names To Get:  1109384


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "10:50am")
#tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-05-19 20:08:46
========================= termTime(day=tomorrow,time=10:50am) =========================
   ====> Terminate Time Set To 2022-05-20 10:50:00 <====
   ====> Will Terminate Process 14 Hours and 41 Minutes From Now
Getting Karelia Cadavid (0003055289)                      True
Getting Aino Karelia (0003020357)                         True
Getting Bill "Sauce Boss" Wharton (0000068612)            True
Getting Sauce Tha Boss (0002671461)                       True
Getting James Archie Worley (0002273802)                  True
Getting Ten Grand (0000018971)                            True
Getting The Vidablue (0001586097)                         True
Getting Vida Dianne Beal (0003088701)                     True
Getting The Blue Voodoo (0000428703)                      True
Getting Blue Foot (0000760401)                            True
Getting Blue Food (0001247257)                            True
Getting Blue Void (

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Mike Marsk (0003338519)                           True
Getting Mike Marasch (0004006162)                         True
Getting Satomi Matsuzaki (0001990352)                     True
Getting Satomi Fukami (0001175320)                        True
Getting Satomi Shiga (0001357172)                         True
Getting Satomi Takasugi (0001555378)                      True
Getting Satomi Endo (0001600084)                          True
Getting Satomi Arimori (0001654003)                       True
Getting Satomi Yamasaki (0001866547)                      True
Getting Satomi Kohrogi (0001910691)                       True
Getting Satomi Takada (0001926223)                        True
Getting Satomi Kawakami (0001948894)                      True
Getting Satomi Sukami (0001949209)                        True
Getting Satomi Tanigawa (0001975778)                      True
Getting Satomi Yanagisawa (0001982455)                    True
Getting Satomi Shirata (0002094301)                    

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Joe Moss and His Orchestra (0002722368)           True
Getting Modern Art Orchestra (0001496536)                 True
Getting Modern Art Sextet (0002725214)                    True
Getting The Modern Art Septet (0003000822)                True
Getting Modern Art Trio (0003268120)                      True
Getting Art Shryer's Modern Jewish Orchestra (0000604199) True
Getting Art Shryer Modern Jewish Orchestra (0002002080)   True
Getting Modern Arts Guitar Quartet (0002597754)           True
Getting Ska Boys (0000747036)                             True
Getting Sky Boys (0001526386)                             True
Getting Sick Boys (0002070756)                            True
Getting The Sico Boys (0002364311)                        True
Getting Boys Too Sick (0001559456)                        True
Getting The Grey Sky Boys (0002115158)                    True
Getting Apres Ski Boys (0002996793)                       True
Getting Soggy Matzoh Boys (0000951852)                 

In [ ]:
from lib.allmusic import moveLocalFiles

In [ ]:
moveLocalFiles()